# Ensemble Models for Loan Default Prediction

This notebook develops and evaluates:
- Logistic Regression
- Random Forest
- XGBoost
- Voting Ensemble
- Stacking Ensemble

The models are trained using:
- selected features
- SMOTE-ENN balanced training data
- preprocessing artifacts generated in previous stages

## Load Selected Feature Datasets

In [8]:
import numpy as np
# Load selected feature datasets generated after preprocessing and feature selection
X_train_selected = np.load("X_train_selected.npy")
y_train_smoteenn = np.load("y_train_smoteenn.npy")

X_test_selected = np.load("X_test_selected.npy")
y_test = np.load("y_test.npy")

print(X_train_selected.shape)
print(X_test_selected.shape)

(42283, 57)
(20000, 57)


## Logistic Regression Model

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

lr = LogisticRegression(max_iter=1000, n_jobs=-1)
lr.fit(X_train_selected, y_train_smoteenn) # Train Logistic Regression model
# Generate predictions and probability scores
y_pred_lr = lr.predict(X_test_selected)
y_prob_lr = lr.predict_proba(X_test_selected)[:, 1]

print("Logistic Regression")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression
              precision    recall  f1-score   support

           0       0.86      0.97      0.91      9492
           1       0.96      0.85      0.91     10508

    accuracy                           0.91     20000
   macro avg       0.91      0.91      0.91     20000
weighted avg       0.91      0.91      0.91     20000

ROC-AUC: 0.9735072918576596


## Random Forest Model

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
# Train Random Forest classifier
rf.fit(X_train_selected, y_train_smoteenn)

y_pred_rf = rf.predict(X_test_selected)
y_prob_rf = rf.predict_proba(X_test_selected)[:, 1]

print("Random Forest")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest
              precision    recall  f1-score   support

           0       0.90      0.97      0.93      9492
           1       0.97      0.90      0.93     10508

    accuracy                           0.93     20000
   macro avg       0.93      0.94      0.93     20000
weighted avg       0.94      0.93      0.93     20000

ROC-AUC: 0.9737225273028587


## XGBoost Model

In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
# Train XGBoost classifier
xgb.fit(X_train_selected, y_train_smoteenn)

y_pred_xgb = xgb.predict(X_test_selected)
y_prob_xgb = xgb.predict_proba(X_test_selected)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      9492
           1       0.97      0.89      0.93     10508

    accuracy                           0.93     20000
   macro avg       0.93      0.93      0.93     20000
weighted avg       0.93      0.93      0.93     20000

ROC-AUC: 0.9789291437054118


## Voting Ensemble

In [12]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Soft Voting Ensemble
# Combine Random Forest and XGBoost using soft voting
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', xgb)
    ],
    voting='soft'
)

# Train
voting_clf.fit(X_train_selected, y_train_smoteenn)

# Predict
y_pred_vote = voting_clf.predict(X_test_selected)
y_prob_vote = voting_clf.predict_proba(X_test_selected)[:, 1]

# Evaluation
print("Soft Voting Ensemble Results")
print(classification_report(y_test, y_pred_vote))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_vote))

Soft Voting Ensemble Results
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      9492
           1       0.97      0.89      0.93     10508

    accuracy                           0.93     20000
   macro avg       0.93      0.93      0.93     20000
weighted avg       0.93      0.93      0.93     20000

ROC-AUC: 0.9761064593733171


## Stacking Ensemble

In [13]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Build stacking ensemble using Logistic Regression as meta-learner
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', xgb)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    passthrough=False
)

# Train stacking ensemble model

stacking_clf.fit(X_train_selected, y_train_smoteenn)

# Predict
y_pred_stack = stacking_clf.predict(X_test_selected)
y_prob_stack = stacking_clf.predict_proba(X_test_selected)[:, 1]

# Evaluation
print("Stacking Ensemble Results")
print(classification_report(y_test, y_pred_stack))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_stack))

Stacking Ensemble Results
              precision    recall  f1-score   support

           0       0.90      0.97      0.93      9492
           1       0.97      0.90      0.93     10508

    accuracy                           0.93     20000
   macro avg       0.93      0.94      0.93     20000
weighted avg       0.94      0.93      0.93     20000

ROC-AUC: 0.9751769706976612


## Load Preprocessing Artifacts

In [14]:
import joblib

# Load preprocessing and feature selection artifacts for inference pipeline
preprocess = joblib.load("preprocess_pipeline.pkl")
# Load selected feature indices
selected_feature_indices = joblib.load("selected_feature_indices.pkl")

# Load raw feature columns
raw_feature_columns = joblib.load("raw_feature_columns.pkl")

## Save Final Stacking Ensemble Model

In [15]:
joblib.dump(stacking_clf, "final_stacking_model.pkl")

['final_stacking_model.pkl']

## Validate Saved Artifacts

In [16]:
print(len(selected_feature_indices))        # should be 76
print(type(preprocess))
print(len(raw_feature_columns))

57
<class 'sklearn.compose._column_transformer.ColumnTransformer'>
118
